<a href="https://colab.research.google.com/github/souha24-ctrl/discursive-stance-analyzer/blob/main/stance_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lightweight Discursive Stance Analyzer

This notebook segments long-form argumentative text into sentence-level propositions and classifies each proposition as **supporting**, **opposing**, or **neutral** toward a target premise.

## Research question

Can a lightweight zero-shot NLP pipeline identify the argumentative stance of individual statements toward a policy claim?

## Pipeline

1. Clean raw text.
2. Segment text into sentences using spaCy.
3. Classify each sentence against a target premise using `facebook/bart-large-mnli`.
4. Evaluate predictions on a labeled dataset.
5. Discuss limitations and possible improvements.

In [1]:
# Install core libraries for segmentation, classification, evaluation, and visualization
!pip install -q spacy transformers torch pandas scikit-learn matplotlib seaborn

# Download spaCy's small English model
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
import torch
import transformers
import pandas as pd
import sklearn

print("spaCy:", spacy.__version__)
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("GPU available:", torch.cuda.is_available())

spaCy: 3.8.16
Transformers: 5.20.2
Torch: 2.11.0+cu130
Pandas: 2.2.3
scikit-learn: 1.6.1
GPU available: True


In [3]:
import re
import spacy

# Load the small English spaCy pipeline
nlp = spacy.load("en_core_web_sm")


def clean_text(text: str) -> str:
    """Normalize whitespace and remove URLs and excessive special characters."""
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s.,;:!?()\"'-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def segment_text(text: str) -> list[str]:
    """Clean a long text and return sentence-level propositions."""
    cleaned_text = clean_text(text)
    doc = nlp(cleaned_text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


sample_text = """
Universal basic income could reduce poverty by giving people a guaranteed income floor.
However, critics argue that it may be too expensive for governments to sustain.
Some studies suggest that direct cash transfers can improve financial stability.
Others believe that such programs could reduce incentives to work.
"""

sentences = segment_text(sample_text)

for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")

1. Universal basic income could reduce poverty by giving people a guaranteed income floor.
2. However, critics argue that it may be too expensive for governments to sustain.
3. Some studies suggest that direct cash transfers can improve financial stability.
4. Others believe that such programs could reduce incentives to work.


In [4]:
import torch
from transformers import pipeline

# Load once; do not reload this for every sentence

# Automatically use GPU if available, otherwise use CPU
device = 0 if torch.cuda.is_available() else -1

print("Using device:", "GPU" if device == 0 else "CPU")

stance_classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

CANDIDATE_LABELS = ["support", "oppose", "neutral"]

Using device: GPU


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [5]:
def classify_stance(
    sentences: list[str],
    target_premise: str,
    batch_size: int = 4
) -> list[dict]:
    """
    Classify each sentence's stance toward a target premise.

    Returns a list of dictionaries:
    {
        "sentence": ...,
        "stance": ...,
        "confidence": ...
    }
    """
    results = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]

        outputs = stance_classifier(
            batch,
            candidate_labels=CANDIDATE_LABELS,
            multi_label=False
        )

        # When one text is provided, Transformers may return a dict instead of a list
        if isinstance(outputs, dict):
            outputs = [outputs]

        for sentence, output in zip(batch, outputs):
            results.append({
                "sentence": sentence,
                "stance": output["labels"][0],
                "confidence": round(float(output["scores"][0]), 4)
            })

    return results


target_premise = "Universal basic income is beneficial"

stance_results = classify_stance(sentences, target_premise)

for result in stance_results:
    print(f"[{result['stance'].upper()}] {result['confidence']:.2f} — {result['sentence']}")

[SUPPORT] 0.93 — Universal basic income could reduce poverty by giving people a guaranteed income floor.
[OPPOSE] 0.83 — However, critics argue that it may be too expensive for governments to sustain.
[SUPPORT] 0.85 — Some studies suggest that direct cash transfers can improve financial stability.
[OPPOSE] 0.91 — Others believe that such programs could reduce incentives to work.


In [6]:
results_df = pd.DataFrame(stance_results)
results_df

,sentence,stance,confidence
0,Universal basic income could reduce poverty by...,support,0.9260
1,"However, critics argue that it may be too expe...",oppose,0.8301
2,Some studies suggest that direct cash transfer...,support,0.8540
3,Others believe that such programs could reduce...,oppose,0.9105
